# Cross-Domain Review Sentiment Classification

This notebook implements Option 2 (Review Sentiment Analysis) for ARTI 404. It trains a binary sentiment classifier (Positive/Negative) on Amazon product and IMDb movie reviews, then evaluates it on an in-domain held-out test and on a cross-domain Yelp restaurant test.

Experimental question: which method performs better on restaurant reviews when trained on product and movie reviews? The main judgment is Macro F1 on the Yelp test, with Accuracy reported alongside; the in-domain test is a supporting comparison only. Two methods are compared: TF-IDF + Logistic Regression (Method 1) and frozen MiniLM sentence embeddings + Logistic Regression (Method 2). Settings for both methods are selected on Development data only; neither test set is used for tuning.

## Setup

Import the libraries used across the notebook.

In [1]:
import platform
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentence_transformers
import sklearn
import torch

C:\Users\moham\Desktop\nlp project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fix the random seed so splits, shuffling and model fitting are reproducible.

In [2]:
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

Define relative project paths and create the output folders if they do not already exist. The notebook is expected to run from the project root, so every path here is relative rather than a path tied to one member's machine.

In [3]:
PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS = PROJECT_ROOT / "results"
MODELS = PROJECT_ROOT / "models"

for path in (DATA_RAW, DATA_PROCESSED, RESULTS, MODELS):
    path.mkdir(parents=True, exist_ok=True)

Record the exact package versions used in this run, so the environment can be reproduced later.

In [4]:
print("Python:", platform.python_version())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("torch:", torch.__version__)

Python: 3.12.10
numpy: 2.5.4
pandas: 3.1.1
scikit-learn: 1.9.1
sentence-transformers: 6.2.0
torch: 2.15.0+cpu


## Data

Download the UCI Sentiment Labelled Sentences archive into `data/raw/` if it is not already there, and extract it. The download is skipped when the three required files are already present; `__MACOSX` entries added by macOS zip tools are ignored everywhere below.

In [5]:
import datetime
import json
import urllib.request
import zipfile

DATA_URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"
RAW_ZIP_PATH = DATA_RAW / "sentiment+labelled+sentences.zip"
DOWNLOAD_INFO_PATH = DATA_RAW / "download_info.json"
REQUIRED_FILES = ["amazon_cells_labelled.txt", "imdb_labelled.txt", "yelp_labelled.txt"]


def find_file(filename):
    matches = [p for p in DATA_RAW.rglob(filename) if "__MACOSX" not in p.parts]
    return matches[0] if matches else None


existing_files = {name: find_file(name) for name in REQUIRED_FILES}

if all(existing_files.values()):
    print("Required data files already present in data/raw/, skipping download.")
    if DOWNLOAD_INFO_PATH.exists():
        recorded = json.loads(DOWNLOAD_INFO_PATH.read_text())
        print("Recorded download date:", recorded["download_date"])
    else:
        print("Recorded download date: unknown (no download_info.json found).")
else:
    download_date = datetime.date.today().isoformat()
    urllib.request.urlretrieve(DATA_URL, RAW_ZIP_PATH)
    with zipfile.ZipFile(RAW_ZIP_PATH) as archive:
        archive.extractall(DATA_RAW)
    DOWNLOAD_INFO_PATH.write_text(
        json.dumps({"source": DATA_URL, "download_date": download_date}, indent=2)
    )
    existing_files = {name: find_file(name) for name in REQUIRED_FILES}
    print(f"Downloaded and extracted on {download_date}.")

print("\nFiles found in data/raw/ (excluding __MACOSX):")
for path in sorted(DATA_RAW.rglob("*")):
    if path.is_file() and "__MACOSX" not in path.parts:
        print(" -", path.relative_to(PROJECT_ROOT))

missing = [name for name, path in existing_files.items() if path is None]
if missing:
    raise FileNotFoundError(f"Required file(s) not found after download: {missing}")

Required data files already present in data/raw/, skipping download.
Recorded download date: 2026-11-11

Files found in data/raw/ (excluding __MACOSX):
 - data\raw\download_info.json
 - data\raw\sentiment labelled sentences\.DS_Store
 - data\raw\sentiment labelled sentences\amazon_cells_labelled.txt
 - data\raw\sentiment labelled sentences\imdb_labelled.txt
 - data\raw\sentiment labelled sentences\readme.txt
 - data\raw\sentiment labelled sentences\yelp_labelled.txt
 - data\raw\sentiment+labelled+sentences.zip


Read each source file line by line and split on the last tab, since the label is always the final field and the review text itself may contain other whitespace. Lines that fail to parse are reported, not dropped silently. Cleaning is limited to Unicode normalisation and whitespace collapsing so negation words are preserved and the stored text is not lowercased.

In [6]:
import unicodedata


def clean_text(text: str) -> str:
    # NFKC normalises compatibility characters (e.g. full-width punctuation);
    # whitespace is then collapsed without touching negation words.
    normalized = unicodedata.normalize("NFKC", text)
    return " ".join(normalized.split())


def load_data(path: Path, source: str) -> tuple[pd.DataFrame, list[str]]:
    rows = []
    invalid_lines = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            line = raw_line.rstrip("\n").rstrip("\r")
            if not line.strip():
                invalid_lines.append(f"{source} line {line_number}: empty line")
                continue
            if "\t" not in line:
                invalid_lines.append(f"{source} line {line_number}: no tab separator")
                continue
            text_part, _, label_part = line.rpartition("\t")
            text = clean_text(text_part)
            label_part = label_part.strip()
            if label_part not in ("0", "1"):
                invalid_lines.append(
                    f"{source} line {line_number}: invalid label {label_part!r}"
                )
                continue
            if not text:
                invalid_lines.append(f"{source} line {line_number}: empty text")
                continue
            rows.append(
                {
                    "example_id": f"{source}_{line_number:04d}",
                    "source": source,
                    "text": text,
                    "label": int(label_part),
                }
            )
    return pd.DataFrame(rows), invalid_lines

Load all three sources and collect any invalid lines found during parsing.

In [7]:
FILENAME_BY_SOURCE = {
    "amazon": "amazon_cells_labelled.txt",
    "imdb": "imdb_labelled.txt",
    "yelp": "yelp_labelled.txt",
}

frames = []
all_invalid_lines = []
for source, filename in FILENAME_BY_SOURCE.items():
    frame, invalid_lines = load_data(existing_files[filename], source)
    frames.append(frame)
    all_invalid_lines.extend(invalid_lines)

raw_df = pd.concat(frames, ignore_index=True)

print(f"Loaded {len(raw_df)} valid rows from {len(FILENAME_BY_SOURCE)} source files.")
print(f"Invalid lines found: {len(all_invalid_lines)}")
for entry in all_invalid_lines:
    print(" -", entry)

Loaded 3000 valid rows from 3 source files.
Invalid lines found: 0


Counts by source and label, before any duplicate handling.

In [8]:
count_table = (
    raw_df.groupby(["source", "label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "negative_0", 1: "positive_1"})
)
count_table["total"] = count_table["negative_0"] + count_table["positive_1"]
count_table

label,negative_0,positive_1,total
source,,,
amazon,500,500,1000
imdb,500,500,1000
yelp,500,500,1000


Build a `duplicate_key` for auditing only (lowercased, whitespace-collapsed text; punctuation and negation words are kept). The stored `text` column is never changed by this key.

In [9]:
def make_duplicate_key(text: str) -> str:
    return " ".join(text.lower().split())


raw_df["duplicate_key"] = raw_df["text"].apply(make_duplicate_key)

Audit duplicate_key groups across all three sources. A group with conflicting labels is dropped entirely and logged as a conflict. A group sharing one label keeps exactly one row, chosen by a fixed source priority (Yelp, then Amazon, then IMDb, then order of appearance in the source file) so a sentence that occurs in Yelp stays in the cross-domain test source rather than in training. No model output is used in this decision.

In [10]:
SOURCE_PRIORITY = {"yelp": 0, "amazon": 1, "imdb": 2}


def audit_duplicates(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    kept_rows = []
    removed_records = []
    within_source_keys = []
    across_source_keys = []
    conflict_keys = []

    for key, group in df.groupby("duplicate_key", sort=False):
        if len(group) == 1:
            kept_rows.append(group.iloc[0])
            continue

        is_cross_source = group["source"].nunique() > 1

        if group["label"].nunique() > 1:
            conflict_keys.append(key)
            for _, row in group.iterrows():
                removed_records.append(
                    {
                        "example_id": row["example_id"],
                        "source": row["source"],
                        "label": row["label"],
                        "duplicate_key": key,
                        "reason": "conflict",
                    }
                )
            continue

        if is_cross_source:
            across_source_keys.append(key)
        else:
            within_source_keys.append(key)

        ordered_group = group.assign(
            _priority=group["source"].map(SOURCE_PRIORITY)
        ).sort_values(["_priority", "example_id"])
        keep_row = ordered_group.iloc[0].drop(labels="_priority")
        drop_group = ordered_group.iloc[1:]

        kept_rows.append(keep_row)
        for _, row in drop_group.iterrows():
            removed_records.append(
                {
                    "example_id": row["example_id"],
                    "source": row["source"],
                    "label": row["label"],
                    "duplicate_key": key,
                    "reason": "duplicate",
                }
            )

    clean_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    removal_log = pd.DataFrame(
        removed_records, columns=["example_id", "source", "label", "duplicate_key", "reason"]
    )

    report = {
        "duplicate_groups_within_source": len(within_source_keys),
        "duplicate_groups_across_source": len(across_source_keys),
        "conflicting_groups": len(conflict_keys),
        "within_source_keys": within_source_keys,
        "across_source_keys": across_source_keys,
        "conflict_keys": conflict_keys,
    }

    return clean_df, removal_log, report

Run the audit and report the group and row counts.

In [11]:
clean_df, removal_log, duplicate_report = audit_duplicates(raw_df)

rows_removed_as_duplicate = int((removal_log["reason"] == "duplicate").sum())
rows_removed_as_conflict = int((removal_log["reason"] == "conflict").sum())

print("Duplicate groups within a single source:", duplicate_report["duplicate_groups_within_source"])
print("Duplicate groups across sources:", duplicate_report["duplicate_groups_across_source"])
print("Conflicting-label groups (dropped entirely):", duplicate_report["conflicting_groups"])
print("Rows removed as duplicates:", rows_removed_as_duplicate)
print("Rows removed as conflicts:", rows_removed_as_conflict)
print("Rows remaining:", len(clean_df))

Duplicate groups within a single source: 18
Duplicate groups across sources: 1
Conflicting-label groups (dropped entirely): 0
Rows removed as duplicates: 21
Rows removed as conflicts: 0
Rows remaining: 2979


Show a few example duplicate-key groups, and a conflicting-label group if one exists, so the kind of overlap being removed is visible.

In [12]:
example_duplicate_keys = (
    duplicate_report["within_source_keys"] + duplicate_report["across_source_keys"]
)[:5]

print("Example duplicate-key groups (same label, one row kept):")
for key in example_duplicate_keys:
    group = raw_df[raw_df["duplicate_key"] == key]
    print(f"\nduplicate_key: {key!r}")
    print(group[["example_id", "source", "label", "text"]].to_string(index=False))

if duplicate_report["conflict_keys"]:
    print("\n\nExample conflicting-label group (whole group dropped):")
    key = duplicate_report["conflict_keys"][0]
    group = raw_df[raw_df["duplicate_key"] == key]
    print(f"duplicate_key: {key!r}")
    print(group[["example_id", "source", "label", "text"]].to_string(index=False))
else:
    print("\n\nNo conflicting-label groups were found.")

Example duplicate-key groups (same label, one row kept):

duplicate_key: 'works great!.'
 example_id source  label          text
amazon_0019 amazon      1 Works great!.
amazon_0525 amazon      1 Works great!.

duplicate_key: 'very disappointed.'
 example_id source  label               text
amazon_0057 amazon      0 VERY DISAPPOINTED.
amazon_0144 amazon      0 very disappointed.

duplicate_key: 'if you like a loud buzzing to override all your conversations, then this phone is for you!'
 example_id source  label                                                                                       text
amazon_0180 amazon      0 If you like a loud buzzing to override all your conversations, then this phone is for you!
amazon_0745 amazon      0 If you like a loud buzzing to override all your conversations, then this phone is for you!

duplicate_key: "don't buy this product."
 example_id source  label                    text
amazon_0181 amazon      0 Don't buy this product.
amazon_0544 amazo

Save the per-source, per-label audit trail: original count, rows removed as duplicates, rows removed as conflicts, and the final count.

In [13]:
original_counts = raw_df.groupby(["source", "label"]).size().rename("original_count")

removed_duplicate_counts = (
    removal_log[removal_log["reason"] == "duplicate"]
    .groupby(["source", "label"])
    .size()
    .rename("removed_as_duplicate")
)
removed_conflict_counts = (
    removal_log[removal_log["reason"] == "conflict"]
    .groupby(["source", "label"])
    .size()
    .rename("removed_as_conflict")
)
final_counts = clean_df.groupby(["source", "label"]).size().rename("final_count")

data_audit = (
    pd.concat(
        [original_counts, removed_duplicate_counts, removed_conflict_counts, final_counts],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

data_audit.to_csv(RESULTS / "data_audit.csv", index=False)
data_audit

,source,label,original_count,removed_as_duplicate,removed_as_conflict,final_count
0,amazon,0,500,4,0,496
1,amazon,1,500,9,0,491
2,imdb,0,500,2,0,498
3,imdb,1,500,2,0,498
4,yelp,0,500,3,0,497
5,yelp,1,500,1,0,499


Counts by source and label after cleaning.

In [14]:
final_count_table = (
    clean_df.groupby(["source", "label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "negative_0", 1: "positive_1"})
)
final_count_table["total"] = final_count_table["negative_0"] + final_count_table["positive_1"]
final_count_table

label,negative_0,positive_1,total
source,,,
amazon,496,491,987
imdb,498,498,996
yelp,497,499,996


Limitation: the UCI source does not provide the original review ID for each sentence, so sentences drawn from the same original review cannot be guaranteed to stay in the same split. Only exact duplicates, matched after case and whitespace normalisation, are detected and handled above; near-duplicate or paraphrased sentences are not checked automatically.

Split the cleaned data into Training, Development, In-domain Test (Amazon + IMDb) and Cross-domain Test (Yelp). A combined `source_label` key stratifies both splits, and `random_state=42` is used at each step. This exact split is shared by both methods.

In [15]:
from sklearn.model_selection import train_test_split


def make_splits(df: pd.DataFrame, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    in_domain_mask = df["source"].isin(["amazon", "imdb"])
    in_domain_df = df[in_domain_mask].copy()
    yelp_df = df[~in_domain_mask].copy()

    in_domain_strat_key = in_domain_df["source"] + "_" + in_domain_df["label"].astype(str)
    train_df, held_out_df = train_test_split(
        in_domain_df,
        test_size=0.30,
        stratify=in_domain_strat_key,
        random_state=random_state,
    )

    held_out_strat_key = held_out_df["source"] + "_" + held_out_df["label"].astype(str)
    dev_df, test_in_domain_df = train_test_split(
        held_out_df,
        test_size=0.50,
        stratify=held_out_strat_key,
        random_state=random_state,
    )

    train_df = train_df.assign(split="train")
    dev_df = dev_df.assign(split="dev")
    test_in_domain_df = test_in_domain_df.assign(split="test_in_domain")
    test_yelp_df = yelp_df.assign(split="test_yelp")

    return pd.concat(
        [train_df, dev_df, test_in_domain_df, test_yelp_df], ignore_index=True
    )

Run the split.

In [16]:
split_df = make_splits(clean_df)

Verify the split before saving or training anything on it.

In [17]:
assert split_df["example_id"].is_unique, "Some example_id appears in more than one split"

keys_per_split = split_df.groupby("duplicate_key")["split"].nunique()
keys_in_multiple_splits = keys_per_split[keys_per_split > 1]
assert keys_in_multiple_splits.empty, (
    f"duplicate_key values span multiple splits: {keys_in_multiple_splits.index.tolist()}"
)

yelp_outside_test = split_df[
    (split_df["source"] == "yelp") & (split_df["split"] != "test_yelp")
]
assert yelp_outside_test.empty, "Yelp rows found outside the test_yelp split"

labels_per_split = split_df.groupby("split")["label"].nunique()
assert (labels_per_split == 2).all(), (
    f"Some split is missing a label: {labels_per_split.to_dict()}"
)

assert len(split_df) == len(clean_df), (
    f"Split total {len(split_df)} does not match cleaned row count {len(clean_df)}"
)

print("All split integrity checks passed.")
print("Total rows across splits:", len(split_df))

All split integrity checks passed.
Total rows across splits: 2979


Save the split manifest.

In [18]:
split_manifest = split_df[["example_id", "source", "label", "split", "duplicate_key"]]
split_manifest.to_csv(DATA_PROCESSED / "split_manifest.csv", index=False)

Counts per split, source and label, and the total per split.

In [19]:
split_source_label_counts = (
    split_df.groupby(["split", "source", "label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "negative_0", 1: "positive_1"})
)
split_source_label_counts["total"] = (
    split_source_label_counts["negative_0"] + split_source_label_counts["positive_1"]
)
print(split_source_label_counts)

split_totals = split_df.groupby("split").size().rename("total_rows")
print("\nTotal rows per split:")
print(split_totals)
print("\nTotal rows across all splits:", len(split_df))

label                  negative_0  positive_1  total
split          source                               
dev            amazon          74          74    148
               imdb            75          74    149
test_in_domain amazon          75          73    148
               imdb            75          75    150
test_yelp      yelp           497         499    996
train          amazon         347         344    691
               imdb           348         349    697

Total rows per split:
split
dev                297
test_in_domain     298
test_yelp          996
train             1388
Name: total_rows, dtype: int64

Total rows across all splits: 2979


The test sets (`test_in_domain` and `test_yelp`) are frozen from this point onward and are never used for tuning. Both Method 1 and Method 2 are trained and selected on the exact same `train` and `dev` splits above.

## Method 1

TF-IDF turns each review into a vector of word weights learned from the training text only. Logistic Regression then learns which weighted words point to Positive or Negative.

`train_tfidf` builds an sklearn Pipeline so TF-IDF is fit on the given training text only; the same vocabulary and weights are then reused unchanged whenever the pipeline transforms development or test text.

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def train_tfidf(train_texts, train_labels, ngram_range, C):
    pipeline = Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    stop_words=None,
                    min_df=1,
                    ngram_range=ngram_range,
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    C=C, solver="lbfgs", max_iter=2000, random_state=RANDOM_STATE
                ),
            ),
        ]
    )
    pipeline.fit(train_texts, train_labels)
    return pipeline

A scoring helper shared by both methods: Accuracy and Macro F1 over the fixed label set.

In [21]:
from sklearn.metrics import accuracy_score, f1_score


def score(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=[0, 1], average="macro", zero_division=0)
    return accuracy, macro_f1

Grid search six candidates on Development only, in the fixed declared order: `ngram_range` (1, 1) then (1, 2); `C` in {0.1, 1, 10}. Each candidate is fit on Training only and scored on Development; any `ConvergenceWarning` is caught and reported rather than silenced. Test splits are not touched in this cell.

In [22]:
import warnings

from sklearn.exceptions import ConvergenceWarning

TFIDF_GRID_NGRAMS = [(1, 1), (1, 2)]
TFIDF_GRID_C = [0.1, 1, 10]

train_texts = split_df.loc[split_df["split"] == "train", "text"].tolist()
train_labels = split_df.loc[split_df["split"] == "train", "label"].tolist()
dev_texts = split_df.loc[split_df["split"] == "dev", "text"].tolist()
dev_labels = split_df.loc[split_df["split"] == "dev", "label"].tolist()

tfidf_dev_records = []
tfidf_fitted_pipelines = {}

for ngram_range in TFIDF_GRID_NGRAMS:
    for C in TFIDF_GRID_C:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            pipeline = train_tfidf(train_texts, train_labels, ngram_range, C)

        convergence_messages = [
            str(w.message) for w in caught if issubclass(w.category, ConvergenceWarning)
        ]
        if convergence_messages:
            print(f"ConvergenceWarning for ngram_range={ngram_range}, C={C}:")
            for message in convergence_messages:
                print(" -", message)

        dev_predictions = pipeline.predict(dev_texts)
        dev_accuracy, dev_macro_f1 = score(dev_labels, dev_predictions)

        tfidf_dev_records.append(
            {
                "method": "TF-IDF_LR",
                "ngram_range": ngram_range,
                "C": C,
                "dev_accuracy": dev_accuracy,
                "dev_macro_f1": dev_macro_f1,
            }
        )
        tfidf_fitted_pipelines[(ngram_range, C)] = pipeline

Select the winner using the declared tie-break policy: highest Development Macro F1, then higher Development accuracy, then smaller `C`, then unigrams, then first in the declared grid order. The results stay in a DataFrame here; they are combined with Method 2's results and written to `results/dev_results.csv` once in the Results section, so re-running the notebook does not duplicate rows.

In [23]:
tfidf_dev_results = pd.DataFrame(tfidf_dev_records)
tfidf_dev_results["grid_order"] = range(len(tfidf_dev_results))
tfidf_dev_results["is_unigram"] = tfidf_dev_results["ngram_range"] == (1, 1)

tfidf_ranked = tfidf_dev_results.sort_values(
    by=["dev_macro_f1", "dev_accuracy", "C", "is_unigram", "grid_order"],
    ascending=[False, False, True, False, True],
).reset_index(drop=True)

tfidf_winner = tfidf_ranked.iloc[0]
tfidf_dev_results["selected"] = (
    tfidf_dev_results["grid_order"] == tfidf_winner["grid_order"]
)
tfidf_dev_results = tfidf_dev_results[
    ["method", "ngram_range", "C", "dev_accuracy", "dev_macro_f1", "selected"]
]

Display the Development results and the selected settings.

In [24]:
print(tfidf_dev_results)

print("\nSelected TF-IDF_LR settings:")
print(f"ngram_range={tfidf_winner['ngram_range']}, C={tfidf_winner['C']}")
print(
    f"dev_accuracy={tfidf_winner['dev_accuracy']:.3f}, "
    f"dev_macro_f1={tfidf_winner['dev_macro_f1']:.3f}"
)

tfidf_model = tfidf_fitted_pipelines[(tfidf_winner["ngram_range"], tfidf_winner["C"])]

      method ngram_range     C  dev_accuracy  dev_macro_f1  selected
0  TF-IDF_LR      (1, 1)   0.1      0.771044      0.770833     False
1  TF-IDF_LR      (1, 1)   1.0      0.784512      0.783961     False
2  TF-IDF_LR      (1, 1)  10.0      0.821549      0.821257      True
3  TF-IDF_LR      (1, 2)   0.1      0.791246      0.791130     False
4  TF-IDF_LR      (1, 2)   1.0      0.781145      0.780787     False
5  TF-IDF_LR      (1, 2)  10.0      0.791246      0.790959     False

Selected TF-IDF_LR settings:
ngram_range=(1, 1), C=10.0
dev_accuracy=0.822, dev_macro_f1=0.821


## Method 2

A frozen pretrained sentence encoder (all-MiniLM-L6-v2) turns each review into a 384-number vector that captures meaning. Logistic Regression then learns Positive/Negative from those vectors. The encoder itself is never fine-tuned.

Load the frozen encoder on CPU and record its maximum sequence length.

In [25]:
from sentence_transformers import SentenceTransformer

minilm_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
print("max_seq_length:", minilm_encoder.max_seq_length)

C:\Users\moham\Desktop\nlp project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\moham\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\moham\Desktop\nlp project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\moham\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]

max_seq_length: 256


`encode_texts` wraps the encoder call used everywhere embeddings are needed.

In [26]:
def encode_texts(model, texts):
    return model.encode(
        texts,
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=False,
    )

Encode train, dev, in-domain test and Yelp test once each. Encoding the test sets here is only feature extraction for later use; nothing in this section looks at a test label or uses a test score for any decision.

In [27]:
test_in_domain_texts = split_df.loc[split_df["split"] == "test_in_domain", "text"].tolist()
test_yelp_texts = split_df.loc[split_df["split"] == "test_yelp", "text"].tolist()

train_embeddings = encode_texts(minilm_encoder, train_texts)
dev_embeddings = encode_texts(minilm_encoder, dev_texts)
test_in_domain_embeddings = encode_texts(minilm_encoder, test_in_domain_texts)
test_yelp_embeddings = encode_texts(minilm_encoder, test_yelp_texts)

named_embeddings = {
    "train": train_embeddings,
    "dev": dev_embeddings,
    "test_in_domain": test_in_domain_embeddings,
    "test_yelp": test_yelp_embeddings,
}

for name, embeddings in named_embeddings.items():
    assert embeddings.shape[1] == 384, (
        f"{name} embeddings have unexpected dimension {embeddings.shape[1]}"
    )
    print(f"{name}: {embeddings.shape}")

train: (1388, 384)
dev: (297, 384)
test_in_domain: (298, 384)
test_yelp: (996, 384)


Count texts whose tokenized length exceeds the encoder's `max_seq_length`, per split, using the model's own tokenizer. This is reported only; nothing is redesigned around it.

In [28]:
def count_long_texts(model, texts):
    token_counts = [len(model.tokenizer.encode(text)) for text in texts]
    return sum(count > model.max_seq_length for count in token_counts)


named_texts = {
    "train": train_texts,
    "dev": dev_texts,
    "test_in_domain": test_in_domain_texts,
    "test_yelp": test_yelp_texts,
}

print(f"Texts exceeding max_seq_length ({minilm_encoder.max_seq_length} tokens):")
for split_name, texts in named_texts.items():
    long_count = count_long_texts(minilm_encoder, texts)
    print(f"{split_name}: {long_count} of {len(texts)}")

Texts exceeding max_seq_length (256 tokens):
train: 0 of 1388
dev: 0 of 297
test_in_domain: 0 of 298


test_yelp: 0 of 996


`train_minilm_classifier` fits Logistic Regression on embeddings only; the encoder that produced them stays frozen.

In [29]:
def train_minilm_classifier(train_embeddings, train_labels, C):
    classifier = LogisticRegression(
        C=C, solver="lbfgs", max_iter=2000, random_state=RANDOM_STATE
    )
    classifier.fit(train_embeddings, train_labels)
    return classifier

Grid search three candidates on Development only, in the declared order `C` in {0.1, 1, 10}. Test embeddings are not scored anywhere in this cell.

In [30]:
MINILM_GRID_C = [0.1, 1, 10]

minilm_dev_records = []
minilm_fitted_classifiers = {}

for C in MINILM_GRID_C:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        classifier = train_minilm_classifier(train_embeddings, train_labels, C)

    convergence_messages = [
        str(w.message) for w in caught if issubclass(w.category, ConvergenceWarning)
    ]
    if convergence_messages:
        print(f"ConvergenceWarning for C={C}:")
        for message in convergence_messages:
            print(" -", message)

    dev_predictions = classifier.predict(dev_embeddings)
    dev_accuracy, dev_macro_f1 = score(dev_labels, dev_predictions)

    minilm_dev_records.append(
        {
            "method": "MiniLM_LR",
            "ngram_range": "",
            "C": C,
            "dev_accuracy": dev_accuracy,
            "dev_macro_f1": dev_macro_f1,
        }
    )
    minilm_fitted_classifiers[C] = classifier

Select the winner with the same tie-break policy: highest Development Macro F1, then higher Development accuracy, then smaller `C`, then first in the declared grid order. Combine with Method 1's results and write `results/dev_results.csv` once, overwriting it, so re-running the notebook does not duplicate rows.

In [31]:
minilm_dev_results = pd.DataFrame(minilm_dev_records)
minilm_dev_results["grid_order"] = range(len(minilm_dev_results))

minilm_ranked = minilm_dev_results.sort_values(
    by=["dev_macro_f1", "dev_accuracy", "C", "grid_order"],
    ascending=[False, False, True, True],
).reset_index(drop=True)

minilm_winner = minilm_ranked.iloc[0]
minilm_dev_results["selected"] = (
    minilm_dev_results["grid_order"] == minilm_winner["grid_order"]
)
minilm_dev_results = minilm_dev_results[
    ["method", "ngram_range", "C", "dev_accuracy", "dev_macro_f1", "selected"]
]

dev_results = pd.concat([tfidf_dev_results, minilm_dev_results], ignore_index=True)
dev_results.to_csv(RESULTS / "dev_results.csv", index=False)

Display the Development results and the selected settings.

In [32]:
print(minilm_dev_results)

print("\nSelected MiniLM_LR settings:")
print(f"C={minilm_winner['C']}")
print(
    f"dev_accuracy={minilm_winner['dev_accuracy']:.3f}, "
    f"dev_macro_f1={minilm_winner['dev_macro_f1']:.3f}"
)

minilm_model = minilm_fitted_classifiers[minilm_winner["C"]]

      method ngram_range     C  dev_accuracy  dev_macro_f1  selected
0  MiniLM_LR               0.1      0.915825      0.915821      True
1  MiniLM_LR               1.0      0.912458      0.912457     False
2  MiniLM_LR              10.0      0.909091      0.909054     False

Selected MiniLM_LR settings:
C=0.1
dev_accuracy=0.916, dev_macro_f1=0.916
